## Snowflake connection

In order to connect to Snowflake, we heve to introduce manually the credentials and the python library `snowflake-snowpark-python` is needed. 

In [ ]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col

I will use a **environment variables**, in order to safely sign in to Snowflake and to load my Claude API.

In [3]:
# Change folder to current directory of the Jupyter Notebook
current_path=os.getcwd()
os.chdir(current_path)

# Load environment variables from .env file
load_dotenv(override=True) # override=True para que no haya conflictos con la Claude API Key en el .env 

True

Connecting to Snowflake and running an example query for testing the successful connection.

In [3]:
# Connection parameters for establishing a Snowpark session
# These environment variables must be set with real credentials before running
connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
    "schema": os.getenv("SNOWFLAKE_SCHEMA"), 
    # "passcode": 901937 # MFA passcode, this is optional if you have an autheticator for your Snowflake Account
}

# Create a Snowpark session using the connection parameters
session = Session.builder.configs(connection_params).create()

# Create a DataFrame reference to the menu table (lazy evaluation – no data fetched yet)
df = session.table("TASTY_BYTES.raw_pos.menu")

# Apply a filter and display results; .show() triggers execution and prints output
df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"MENU_ID"  |"MENU_TYPE_ID"  |"MENU_TYPE"  |"TRUCK_BRAND_NAME"  |"MENU_ITEM_ID"  |"MENU_ITEM_NAME"    |"ITEM_CATEGORY"  |"ITEM_SUBCATEGORY"  |"COST_OF_GOODS_USD"  |"SALE_PRICE_USD"  |"MENU_ITEM_HEALTH_METRICS_OBJ"     |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|10001      |1               |Ice Cream    |Freezing Point      |10              |Lemonade            |Beverage         |Cold Option         |0.6500               |3.5000            |{                                  |
|           |                |             |                    |                |                    |                 

## Testing Claude API connection

In [4]:
import anthropic

client = anthropic.Anthropic()

Testing the Claude API to see if it works correctly.

In [ ]:
message = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=1000,
    messages=[
        {
            'role': 'user',
            'content': 'Describe briefly the Tasty Bytes Dataset provided by Snowflake? Please respond using a maximum of 10 lines.',
        }
    ]
)

for block in message.content:
    if block.type == "text":
        print(block.text)

The **Tasty Bytes** dataset is a fictional, globally distributed food truck company that Snowflake provides for demos, tutorials, and quickstarts. It includes sample data on menus, trucks, locations, point-of-sale orders, and customer loyalty, spanning many cities across multiple countries. Because it's realistic but synthetic, it lets users practice data loading, transformation, analytics, and AI/ML workflows in Snowflake without using real business data.


## Making some tests with Claude API

In [19]:
# functions for multi-turn conversations

def add_user_msg(msgs, txt):
    # Adds an user message to the history
    user_msg = {'role': 'user', 'content':txt}
    msgs.append(user_msg)

def add_assistant_msg(msgs, txt):
    # Adds an assistant (AI) message to the history
    assistant_msg = {'role': 'assistant', 'content':txt}
    msgs.append(assistant_msg)

def chat(msgs, model, max_tokens=1000, system_prompt=None, output_format=None):
    # Creates a chat with the corresponding message history
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': msgs
    }
    if system_prompt:
        params['system'] = system_prompt
    if output_format:
        params['output_config'] = {'format': output_format}

    output = client.messages.create(**params)
    # Joining all text blocks, and checking if the block is a TextBlock
    return "".join(b.text for b in output.content if b.type == "text")

In [ ]:
# Testing multi-turn conversations with Claude API
messages = []

add_user_msg(messages, 'Define what a relational database is in one sentence')

ai_answer = chat(messages, model='claude-sonnet-5-5')

add_assistant_msg(messages, ai_answer)

add_user_msg(messages, 'Write another sentence')

final_answer = chat(messages, model='claude-sonnet-5-5')
final_answer

'A relational database stores information in structured tables that link to one another through primary and foreign keys, which lets you retrieve and combine related data efficiently while keeping it consistent and free of unnecessary duplication.'

## Starting to write the Claude-Snowflake API Pipeline

Re-writing relevant code for the pipliine.

In [1]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum as sum_, avg, count
import numpy as np

import json
from pathlib import Path

import anthropic

# Change folder to current directory of the Jupyter Notebook
current_path=os.getcwd()
os.chdir(current_path)

# Load environment variables from .env file
load_dotenv(override=True) # override=True para que no haya conflictos con la Claude API Key en el .env 

# Connecting to Claude API
client = anthropic.Anthropic()

In [3]:
def add_user_msg(msgs, txt):
    ##### Adds an user message to the history #####
    user_msg = {'role': 'user', 'content':txt}
    msgs.append(user_msg)

def add_assistant_msg(msgs, txt):
    ##### Adds an assistant (AI) message to the history #####
    assistant_msg = {'role': 'assistant', 'content':txt}
    msgs.append(assistant_msg)

def chat(msgs, model, max_tokens=1000, system_prompt=None, output_format=None):
    # Creates a chat with the corresponding message history
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': msgs
    }
    if system_prompt:
        params['system'] = system_prompt
    if output_format:
        params['output_config'] = {'format': output_format}

    output = client.messages.create(**params)
    # Joining all text blocks, and checking if the block is a TextBlock
    return "".join(b.text for b in output.content if b.type == "text")

def chat_and_tokens(msgs, model, max_tokens=1000, use_cache=True, system_prompt=None, output_format=None):
     ##### Creates a chat with the corresponding message history, and outputs the input and output tokens used #####
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': msgs
    }
    if system_prompt:
        if use_cache: # This will execute if cache usage is needed
            params['system'] = [
                {
                    "type": "text",
                    "text": system_prompt,
                    "cache_control": {"type": "ephemeral"}  # Configuring the system prompt to be written in the cache. Lasts 5 minutes by default
                }
            ]
        else: # This will execute if cache usage is not needed
            params['system'] = system_prompt
    if output_format:
        params['output_config'] = {'format': output_format}

    output = client.messages.create(**params)
    # Using an array for saving the tokens used, so for future usages, we can add up the tokens used
    tokens_used = np.array([output.usage.input_tokens, output.usage.output_tokens, output.usage.cache_creation_input_tokens, output.usage.cache_read_input_tokens])
    # Joining all text blocks, and checking if the block is a TextBlock
    return "".join(b.text for b in output.content if b.type == "text"), tokens_used

In this case, I will use the text from ``'claude_prompt_original.md'`` as the system prompt. In this file, all the context and specific instructions for generating the SQL queries based on the Tasty Bytes database documentaation are there.

In [14]:
sql_schema = {
    "type": "json_schema",
    "schema": {
        "anyOf": [
            { # Output Schema in case the user question is simple or doesnt need a clarification
                "type": "object",
                "properties": {
                    "question_type": {"type": "string", "const": "sql"},
                    "query": {
                        "type": "string",
                        "description": "Query SQL ejecutable que responde la pregunta.",
                    },
                    "explanation": {
                        "type": "string",
                        "description": "Explicación breve de la query, máximo 3 oraciones.",
                    },
                },
                "required": ["question_type", "query", "explanation"],
                "additionalProperties": False,
            },
            { # Output schema in case the user question needs a clarification
                "type": "object",
                "properties": {
                    "question_type": {"type": "string", "const": "clarify"},
                    "message": {
                        "type": "string",
                        "description": "Pregunta breve de aclaración para el usuario.",
                    },
                },
                "required": ["question_type", "message"],
                "additionalProperties": False,
            },
        ]
    },
}

# Test Question for Claude  
test = 'What was the average order value (total per order, not per item) for each truck brand in Canada during 2021?'

user_question = f"Write an SQL query in order to answer the user's question about Tasty Bytes: {test}" # User question un natural language
# Later, this variable will recieve the input of the streamlit app, like:
# user_question = input('Ask a question about Tasty Bytes.') 

messages = []
add_user_msg(messages, user_question) # Adding the user question prompt to the message history

system = Path('claude_prompt_original.md').read_text(encoding='utf-8') #Loading the MD file with the system prompt with all the specifications

ai_answer, tokens = chat_and_tokens( # Calling Claude API to write the SQL query
    messages,
    model='claude-sonnet-5-5',
    max_tokens=2000,
    system_prompt=system,
    output_format=sql_schema,
)

input_tok, output_tok, cache_creation_tok, cache_read_tok = tokens

add_assistant_msg(messages, ai_answer) # Saving the AI response in the message history [CHECAR DESPUÉS SI LE METO TODO AL CHAT EN FORMATO JSON O EN OTRO FORMATO] 

result = json.loads(ai_answer) # Converting the string generated by Claude into an JSON
print(result["query"]) # Printing the query
print('\n# of Input Tokens used:',input_tok) # How much input tokens were used?
print('# of Ouput Tokens used:',output_tok) # How much output tokens were used?
print('# of Input Tokens used to create the cache entry:',cache_creation_tok) 
print('# of Input Tokens read from the cache:',cache_read_tok) 

WITH orders AS (SELECT order_id, truck_brand_name, MAX(order_total) AS order_total FROM tasty_bytes.analytics.orders_v WHERE country = 'Canada' AND date >= '2021-01-01' AND date < '2022-01-01' GROUP BY order_id, truck_brand_name) SELECT truck_brand_name, COUNT(*) AS num_orders, AVG(order_total) AS avg_order_value FROM orders GROUP BY truck_brand_name ORDER BY avg_order_value DESC

# of Input Tokens used: 62
# of Ouput Tokens used: 676
# of Input Tokens used to create the cache entry: 6472
# of Input Tokens read from the cache: 0


In [15]:
print(result['explanation'])

Usa la vista ORDERS_V filtrada a Canadá y al año 2021, y colapsa las líneas de cada pedido a una sola fila por order_id para evitar el fan-out de order_total. Luego calcula el promedio del total por pedido para cada marca de camión (truck_brand_name).


Connecting to Snowflake and running the query generated by Claude:

In [16]:
# Connection parameters for establishing a Snowpark session
# These environment variables must be set with real credentials before running
connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
    "schema": os.getenv("SNOWFLAKE_SCHEMA")
}

# Create a Snowpark session using the connection parameters
session = Session.builder.configs(connection_params).create()

In [17]:
# Running the SQL query 
df = session.sql(result["query"]).collect()
df

[Row(TRUCK_BRAND_NAME='Kitakata Ramen Bar', NUM_ORDERS=249358, AVG_ORDER_VALUE=Decimal('60.3789671877')),
 Row(TRUCK_BRAND_NAME="Nani's Kitchen", NUM_ORDERS=651947, AVG_ORDER_VALUE=Decimal('58.5519543767')),
 Row(TRUCK_BRAND_NAME='Cheeky Greek', NUM_ORDERS=819764, AVG_ORDER_VALUE=Decimal('50.6074296993')),
 Row(TRUCK_BRAND_NAME='Smoky BBQ', NUM_ORDERS=241895, AVG_ORDER_VALUE=Decimal('45.0672895264')),
 Row(TRUCK_BRAND_NAME='Revenge of the Curds', NUM_ORDERS=459133, AVG_ORDER_VALUE=Decimal('44.9415332812')),
 Row(TRUCK_BRAND_NAME='Le Coin des Crêpes', NUM_ORDERS=473065, AVG_ORDER_VALUE=Decimal('42.7187595785')),
 Row(TRUCK_BRAND_NAME='The Mac Shack', NUM_ORDERS=903407, AVG_ORDER_VALUE=Decimal('39.3214298760')),
 Row(TRUCK_BRAND_NAME='Tasty Tibs', NUM_ORDERS=242045, AVG_ORDER_VALUE=Decimal('39.2718420128')),
 Row(TRUCK_BRAND_NAME='Better Off Bread', NUM_ORDERS=571451, AVG_ORDER_VALUE=Decimal('39.2698341590')),
 Row(TRUCK_BRAND_NAME='Peking Truck', NUM_ORDERS=540462, AVG_ORDER_VALUE=Decim

## Prompt Engineering

Before procceding, it is worth to point out that, even though we can work with this prompt for the final product, it is not best practice. We need to check if the prompt that we are giving to Claude is good enough in answering similar questions or more complicated ones.

In order to improve the current prompt, we need to apply some prompt engineering techniches:
- **Being clear**: Using simple language, stating what you want explicitly.
- **Being direct**: Using instructions, not questions, and direct action verbs.
- **Being specific**: Providing a lists of steps or a clear guideline of what to do.
- **Providing examples**: Writing down examples of possible user requests and the expected answers.
- **Using XML tags**: Using XML tags that separate distinct portions of the prompt into blocks of text.

In order to verify that these techiniches actually work, I will make some prompt evaluations on 3 different system prompts (not on the promppt itself, since it is always basically the user's question with some added text):

- ``'claude_prompt_original.md'``: As explained before, this is the first version of the system prompt made by myself. It is very simple and straight to the point, although it is not very specific and clear on what Claude needs to do and includes a compact version of the Tasty Bytes database documentation. 

- ``'claude_prompt_improved.md'``: It is a better version of the previous prompt, made by myself. It contains more explanations about each of the 3 typses of users' questions, a clear set of rules for making a proper SQL query style format and some examples for each type of question.

- ``'claude_prompt_fully_improved.md'``: This is the best prompt (in theory) optiimized by Claude itself, using the previous one as the base. It includes XML tags, a new section with wquery business rules for more context, some default values for innecesary aclarations and many more delimiitations for a crearer output.

### Claude API custom functions for SQL query generation

In [4]:
def generate_sql(user_question, system):
    """
    Takes a user question and a system prompt and generates a SQL query using the Claude API.

    Parameters
    ----------
    user_question : str
        Question input by the user.
    system : str
        System prompt for Claude, with all the specifications and database documentation.

    Returns
    ----------
    ai_answer : dict
        Claude's answer as a dict, loaded from a JSON structure.
    tokens : list[int]
        A list of input, output and cache tokens used on the Claude API call. 
        Its structure is ``[input_tok, output_tok, cache_creation_tok, cache_read_tok]``.
    messages : list[dict]
        Message history of user-assistant conversation.
    """
    # Creating a JSON Schema that Claude will use for the output
    sql_schema = {
        "type": "json_schema",
        "schema": {
            "anyOf": [
                { # Output Schema in case the user question is simple or doesnt need a clarification
                    "type": "object",
                    "properties": {
                        "question_type": {"type": "string", "const": "sql"},
                        "query": {
                            "type": "string",
                            "description": "Executable SQL Query that answers the question.",
                        },
                        "explanation": {
                            "type": "string",
                            "description": "Brief explanation of the query of no more than 3 sentences.",
                        },
                    },
                    "required": ["question_type", "query", "explanation"],
                    "additionalProperties": False,
                },
                { # Output schema in case the user question needs a clarification
                    "type": "object",
                    "properties": {
                        "question_type": {"type": "string", "const": "clarify"},
                        "message": {
                            "type": "string",
                            "description": "Brief clarification question for the user.",
                        },
                    },
                    "required": ["question_type", "message"],
                    "additionalProperties": False,
                },
            ]
        },
    }

    prompt = f"Write an SQL query in order to answer the user's question about Tasty Bytes: {user_question}" # User question un natural language
    # Later, this variable will recieve the input of the streamlit app, like:
    # user_question = input('Ask a question about Tasty Bytes.') 

    messages = []
    add_user_msg(messages, prompt) # Adding the user question prompt to the message history

    ai_answer, tokens = chat_and_tokens( # Calling Claude API to write the SQL query
        messages,
        model='claude-sonnet-5-5',
        max_tokens=2000,
        system_prompt=system,
        output_format=sql_schema,
    )
    add_assistant_msg(messages, ai_answer) # Adding the user question prompt to the message history
    ai_answer = json.loads(ai_answer) # Converting the string generated by Claude into an JSON (dict)

    return ai_answer, tokens, messages # The message history will be needed for multi-turn conversations in case an aclaration is needed

In [5]:
def generate_sql_after_aclaration(messages, system, clarif):
    """
    Takes the aclaration question made by Claude, the response of the user and the same system prompt for generating a SQL query using the Claude API,
    after Claude identified that the users question is too vague.

    Parameters
    ----------
    messages : list[dict]
        Message history of user-assistant conversation.
    system : str
        System prompt for Claude, with all the specifications and database documentation.
    clarif : str
        Clarification made by the user after Claude's question

    Returns
    ----------
    ai_answer : str
        Claude's answer as a string, but with a JSON structure.
    tokens : list[int]
        A list of input, output and cache tokens used on the Claude API call. 
        Its structure is ``[input_tokens, output_tokens, cache_creation_tokens, cache_read_tokens]``.
    """
    
    # Creating a JSON Schema that Claude will use for the output
    sql_schema = {
        "type": "json_schema",
        "schema": {
            "anyOf": [
                { # Output Schema in case the user question is simple or doesnt need a clarification
                    "type": "object",
                    "properties": {
                        "question_type": {"type": "string", "const": "sql"},
                        "query": {
                            "type": "string",
                            "description": "Executable SQL Query that answers the question.",
                        },
                        "explanation": {
                            "type": "string",
                            "description": "Brief explanation of the query of no more than 3 sentences.",
                        },
                    },
                    "required": ["question_type", "query", "explanation"],
                    "additionalProperties": False,
                },
                { # Output schema in case the user question needs a clarification
                    "type": "object",
                    "properties": {
                        "question_type": {"type": "string", "const": "clarify"},
                        "message": {
                            "type": "string",
                            "description": "Brief clarification question for the user.",
                        },
                    },
                    "required": ["question_type", "message"],
                    "additionalProperties": False,
                },
            ]
        },
    }

    add_user_msg(messages, f'User Clarification: {clarif}') # Adding the user clarification to the conversation

    prompt = f"Write an SQL query in order to answer the original user's question about Tasty Bytes, taking into account the recent user clarification." 
    add_user_msg(messages, prompt) # Adding the aclaration prompt to the message history

    ai_answer, tokens = chat_and_tokens( # Calling Claude API to write the SQL query
        messages,
        model='claude-sonnet-5-5',
        max_tokens=2000,
        system_prompt=system,
        output_format=sql_schema,
    )
    ai_answer = json.loads(ai_answer) # Converting the string generated by Claude into an JSON (dict)

    return ai_answer, tokens

## Prompt Evaluation

Now, after the SQL query generation code is done, we can do some prompt evaluation, which is basically testing each system prompt made so far on a sample test dataset with different test user questions. In this case, I will test one question for each of the 3 types of questions Claude could receive: 

- **Specific question**: *"What was the average order value (total per order, not per item) for each truck brand in Canada during 2021?"*
- **Vague question with a clear interpretation**: *"Which menu items sell the most?"*
- **Vague question without a clear interpretation**: *"Is the loyalty program working?"*

For the vague question without a clear interpretation, it is expected that Claude will ask for a clarification. In this case, I will interrupt the testing and ask for an input, showing the question asked by Claude (I could give the same answer for each one, but we cannot be sure if Claude will make the same question every single time). Even if the answer won't be the same, I will try to answer in order to get the following:

> "I would like to judge the loyalty program using the average total sales and average number of locations visited per loyalty member, by country."

For comparing each result, we need to calculate an evaluation score. I will use 3 different criteria:
1. Quality score: Evaluating the output quality by using another Claude API call as the judge.
2. Syntax score: Validating that the Snowflake SQL query generated has an correct syntax, using the python library `sqlglot`
3. Cost penalty: Checking how much did the Claude API call costed and reducing the final score based on it

Since the system prompts are longer the more specific they are, I want to check not only the quality and syntax of the output, but also not consuming an exaggerated number of tokens.

In [ ]:
# Making the functions that will score each prompt based on the previous criteria.
import sqlglot
from sqlglot.errors import ParseError, TokenError

def quality_score(test_case, system_prompt, query_output, aclaration_list=None):
    """
    Evaluating the output quality using another Claude API call as the judge.

    Parameters
    ----------
    test_case : str
        The test case original task being evaluated.
    system_prompt : str
        System prompt for Claude used for that output, which will be evaluated.
    query_output : str
        The SQL query output obtained from Claude.
    aclaration_list : list[str]
        Clarification made by the user after Claude's question, containing the aclaration question and the user response.

    Returns
    ----------
    eval_results : dict
        Claude's judge answer with a JSON structure: ``{"strengths": str, "weaknesses": str, "score": int}``
    """

    # Adding the aclaration question and response, in case it applies
    aclaration_str = f'Aclaration question from AI, since the original task was too vague: {aclaration_list[0]} \nResponse from the user: {aclaration_list[1]}' if aclaration_list else '' #

    eval_prompt = f"""
You are an expert SQL code reviewer. Your task is to evaluate the following AI-generated query solution, with a given system prompt and the documentation of the Tasty Bytes Database.
Note: Even though the system prompt mentions that the output should be in a JSON format, I will just provide you just the SQL query, not the complete JSON output.

Original Task:
<task>
Write an SQL query in order to answer the user's question about Tasty Bytes: {test_case}
</task>

Task aclaration (if this is empty, dismiss it):
<aclaration>
{aclaration_str}
</aclaration>

System prompt used:
<system_prompt>
{system_prompt}
</system_prompt>

Solution to Evaluate:
<solution>
{query_output}
</solution>

<rubric>
Combine these criteria equally into one score from 0 to 10:
- Accuracy: tables, columns and joins exist in the documentation; filters, aggregations and grouping match the question.
- Completeness: returns everything the question asks for (metrics, periods, ordering, limits).
- Usefulness: output columns are clear and answer the question directly, with no unnecessary extras.
- Quality: readable, efficient (no needless subqueries or SELECT *), idiomatic Snowflake SQL, a good SQL format style.

Scale: 0 = does not answer the question; 1-3 = major errors; 4-6 = partially correct; 7-8 = correct with minor issues; 9-10 = correct, complete and clean.
Be strict: reserve 9-10 for near-flawless solutions. Syntax is validated separately, so judge the logic.
</rubric>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "score": A number between 0-10, following the rubric

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "score": number
}}
    """

    json_schema = { #schema for the json output of the evaluation
        "type": "json_schema",
        "schema": { 
            "type": "object",
            "properties": {
                "strengths": {"type": "array", "items": {"type": "string"}},
                "weaknesses": {"type": "array", "items": {"type": "string"}},
                "score": {"type": "number"},
            },
            "required": ["strengths", "weaknesses", "score"],
            "additionalProperties": False,
        },
    }

    messages = []
    add_user_msg(messages, eval_prompt)
    eval_text = chat(messages, model='claude-sonnet-5-5', output_format=json_schema) 
    return json.loads(eval_text)

def syntax_score(query_output):
    """
    Validating the output syntaxis using the sqlglot library.

    Parameters
    ----------
    query_output : str
        The SQL query output obtained from Claude.

    Returns
    ----------
    eval_result : int
        Evaluation final score of 0 or 10.
    """

    try:
        # Parse the string specifically using the Snowflake dialect
        sqlglot.parse_one(query_output.strip(), read="snowflake")
        return 10  # Valid syntax
    except (ParseError, TokenError):
        return 0   # Syntax error

For the token cost calculation, I will perform a weighted sum of each type of token used, using the current costs for Sonnet 5.5.

In [7]:
def token_cost(in_tokens, out_tokens, cache_creation_tokens, cache_read_tokens):
    """
    Calculating token cost using the updated base cost of Sonnet 5.5 (from Nov 10th 2026 from https://platform.claude.com/docs/es/about-claude/pricing):
    input tokens: USD 2 / MTok, output tokens: USD 10 / MTok, cache creation input tokens (5 minutes): USD 2.5 / MTok, cache read input tokens: USD 0.1 / MTok

    Parameters
    ----------
    in_tokens : int
        The number of input tokens used.
    out_tokens : int
        The number of output tokens used.
    cache_creation_tokens : int
        The number of input tokens used to create the cache entry.
    cache_read_tokens : int
        The number of input tokens read from the cache.

    Returns
    ----------
    final_cost : float
        Cost in USD of the query.
    """

    if not cache_creation_tokens: # In case it is None when cache is not written
        cache_creation_tokens = 0
    if not cache_read_tokens: # In case it is None when cache is not read
        cache_read_tokens = 0

    
    in_cost = 2/1e6
    out_cost = in_cost*5
    cache_creat_cost = 2.5/1e6
    cache_read_cost = 0.2/1e6

    final_cost = in_tokens*in_cost + out_tokens*out_cost + cache_creation_tokens*cache_creat_cost + cache_read_tokens*cache_read_cost
    return final_cost

**Cálculo de score final**
- For each test case, the average syntax and quality scores will be calculated: Q = (syntax(0/10) + quality(0–10)/2).
- Then, the score and cost will be averaged across all prompts: ``Q̄`` and ``C̄``.
- The final score will include a logarithmic penalty based on the average cost:

``score = Q̄ - log2(C̄ / C_ref)``

``C_ref`` is the cost of your baseline prompt. Therefore, each time a prompt's cost doubles, it loses 1 final score point. In this case, ``C_ref`` will be the smallest cost across all tests.

In [8]:
def final_score(quality_scores, syntax_scores, costs, C_ref):
    """
    Calculating the final score after evaluating a prompt on every test case, using the formula score = Q - log2(C / C_ref).

    Parameters
    ----------
    quality_scores : pd Series Object
        All the quality scores obtained.
    syntax_scores : pd Series Object
        All the syntax scores obtained.
    costs : pd Series Object
        All the costs calculated based on the token costs.
    C_ref : float
        The reference cost. It should be the minimum cost from every prompt and test case runned on the evaluation

    Returns
    ----------
    score : float
        Final score for the prompt.
    Q : float
        Average syntax-quality score.
    penalty : float
        Penalty calculated from average token cost.
    """

    Q_case = (quality_scores + syntax_scores) / 2 # Calculating the avarage score for each test case
    Q = Q_case.mean() # Obtaining the average syntax-quality score
    C = costs.mean() # Calculating the average cost
    penalty = np.log2(C / C_ref)

    score = Q - penalty
    return score, Q, penalty

Finally, we can execute the prompt evaluation, saving the results on a Dataframe and then, calculate each prompt final score:

In [9]:
import uuid
import pandas as pd

# Opening the JSON file with the Evaluation Tests
with open('eval_dataset.json', 'r') as file:
    test_dataset = json.load(file)

# Loading all system prompts that will be tested
system_prompt_names = ['claude_prompt_original.md', 'claude_prompt_improved.md', 'claude_prompt_fully_improved.md']
system_prompts = [
    Path(system_prompt_names[0]).read_text(encoding='utf-8'), # Loading the original system prompt
    Path(system_prompt_names[1]).read_text(encoding='utf-8'), # Loading the system prompt with the improved version I made
    Path(system_prompt_names[2]).read_text(encoding='utf-8') # Loading the system prompt with the improved version made by Claude
]

rows_results = []

for s, sys_prompt in enumerate(system_prompts): #Running the prompt for each system prompt made
    for test in test_dataset: #Running the prompt for each test case
        sys = f"<!-- run:{uuid.uuid4()} -->\n" + sys_prompt # Adding a random UUID to the system prompt, so that the cache resets every test run
        ai_answer, used_tokens, msg_hist = generate_sql(user_question=test['task'], system=sys)
        aclaration_list = None # By default, this will be None
        flag_clarification = 'No'

        if ai_answer['question_type'] == 'clarify': # This will execute only if Claude needs an aclaration from the user
            clarif_question = ai_answer['message']
            # Asking for an input with the user aclaration
            clarification = input(f"Original test question: {test['task']} |  Claude's clarification question: {clarif_question}")
            ai_answer, used_tokens_now = generate_sql_after_aclaration(messages=msg_hist, system=sys, clarif=clarification)
            used_tokens += used_tokens_now
            aclaration_list = [clarif_question, clarification]
            flag_clarification = 'Yes'

        # Calculating the syntax scores
        synt_score = syntax_score(query_output=ai_answer['query'])
        # Calculating the quality scores
        params = {
            'test_case': test['task'],
            'system_prompt': sys_prompt,
            'query_output': ai_answer['query']
        }
        if aclaration_list: # Adding the clarification question and response only if there was a clarication made
            params['aclaration_list'] = aclaration_list
        qual_results = quality_score(**params)
        # Calculating the cost of the Claude API call
        cost = token_cost(*used_tokens)
        # Saving the results of this test on a list
        rows_results.append({'System prompt':system_prompt_names[s], 'Test Case':test['task'], 'Clarification?':flag_clarification, 'Cost (USD)':cost, 'Syntax Score':synt_score, 
                             'Quality Score':qual_results['score'], 'Strengths':qual_results['strengths'], 'Weaknesses':qual_results['weaknesses'], 'SQL output':ai_answer['query']})

# Converting the saved results into a Dataframe
eval_results = pd.DataFrame(rows_results)

ref_cost = eval_results['Cost (USD)'].min() # Using the minimum cost as the reference cost
# After executing every test, we can obtain our reference cost (C_ref) and calculate their final scores
for prompt_name in system_prompt_names:
    prompt_results = eval_results[eval_results['System prompt'] == prompt_name] # Just checking the results of one prompt
    prompt_score, Q, penalty = final_score(quality_scores=prompt_results['Quality Score'], syntax_scores=prompt_results['Syntax Score'], 
                                           costs=prompt_results['Cost (USD)'], C_ref=ref_cost)
    print(prompt_name, '| Avg syntax-quality score:', round(Q,3), '| Cost penalty:', round(-penalty,3), '| Final score:', round(prompt_score,3))

claude_prompt_original.md | Avg syntax-quality score: 8.917 | Cost penalty: -0.067 | Final score: 8.849
claude_prompt_improved.md | Avg syntax-quality score: 8.75 | Cost penalty: -0.56 | Final score: 8.19
claude_prompt_fully_improved.md | Avg syntax-quality score: 9.167 | Cost penalty: -0.788 | Final score: 8.378


In [10]:
eval_results

,System prompt,Test Case,Clarification?,Cost (USD),Syntax Score,Quality Score,Strengths,Weaknesses,SQL output
0,claude_prompt_original.md,What was the average order value (total per or...,No,0.026017,10,8.0,"[Averages order_total at order_header grain, s...",[Includes an unrequested order_count column an...,"WITH brands AS (SELECT DISTINCT menu_type_id, ..."
1,claude_prompt_original.md,Which menu items sell the most?,No,0.023731,10,7.5,[Uses valid tables and columns (order_detail j...,[Scans the 673M-row order_detail table with no...,"SELECT m.menu_item_id, m.menu_item_name, m.tru..."
2,claude_prompt_original.md,Is the loyalty program working?,Yes,0.023214,10,8.0,[Uses the pre-aggregated ANALYTICS.CUSTOMER_LO...,"[Uses the unqualified `country` column, which ...","SELECT country, COUNT(*) AS loyalty_members, A..."
3,claude_prompt_improved.md,What was the average order value (total per or...,No,0.038826,10,7.5,[Uses order_header at order grain with AVG(ord...,"[Filters Canada on truck.country, which is the...","-- One brand per menu_type_id, to avoid fan-ou..."
4,claude_prompt_improved.md,Which menu items sell the most?,No,0.030627,10,7.0,[Uses analytics.orders_v with SUM(quantity) fo...,[Style violations against the system prompt: c...,-- Top 10 menu items by units sold (defaults: ...
5,claude_prompt_improved.md,Is the loyalty program working?,Yes,0.033190,10,8.0,[Uses the correct view (customer_loyalty_metri...,[Violates the system prompt formatting rules: ...,"SELECT country,\n COUNT(customer_id) AS num..."
6,claude_prompt_fully_improved.md,What was the average order value (total per or...,No,0.037814,10,9.0,[Uses analytics.orders_v and computes average ...,[Uses the output alias avg_order_value but ORD...,"SELECT\n o.truck_brand_name,\n SUM(o.pri..."
7,claude_prompt_fully_improved.md,Which menu items sell the most?,No,0.038112,10,8.0,[Uses analytics.orders_v with fully qualified ...,"[Ranks by revenue, while 'sell the most' most ...","SELECT\n o.menu_item_name,\n SUM(o.price..."
8,claude_prompt_fully_improved.md,Is the loyalty program working?,Yes,0.044344,10,8.0,[Uses the preferred view customer_loyalty_metr...,"[Country is the customer's residence, not purc...","SELECT\n c.country,\n COUNT(DISTINCT c.c..."


We can see that the model with the best score was, surprisingly, the most simple prompt. Even though the fully improved prompt has the highest average syntax-quality score, the cost penalty is very high, given the length increase of the system prompt. In contrast the imporved prompt had a worst score in general, so the increment on cost is not worth it. 


Given that the increase on quality is not too big, I decided to include both prompts on the final app, by allowing the user to switch between 2 different modes:
- **Precission mode**: This mode guarantees the best output, at the cost of more tokens used by Claude API. It will use the system prompt ``'claude_prompt_fully_improved.md'``.
- **Token-saving mode**: Thid mode sacrifices some accuracy on the final output, but consumes much Claude API less tokens. It will use the system prompt ``'claude_prompt_original.md'``